
### Layer 2 — The Loan Itself
*Tables: loans + countries*

> Are loans actually being used well — and are the poorest countries
> ending up with more debt than benefit?

**Questions:**
1. What is the ratio of **disbursed vs original principal** — are loans being fully used?
2. Which countries have the highest **cancellation rates** — money borrowed but never used?
3. Which **income groups** carry the most debt relative to what they actually received?
4. What does the **repayment landscape** look like — how many loans are fully repaid vs still active vs cancelled?
5. Are **soft loan countries** (IDA/XDR) borrowing more than they can handle?


In [1]:
import pandas as pd

loans = pd.read_csv("../data/cleaned/loans_clean.csv")
countries = pd.read_csv("../data/cleaned/countries_clean.csv")

print(f"loans: {loans.shape}")
print(f"countries: {countries.shape}")

loans: (11236, 30)
countries: (211, 10)


In [2]:
loans_enriched = loans.merge(
    countries[["country_code_iso3", "country_code_iso2", "income_level", "lending_type", "region"]],
    left_on="country_code",
    right_on="country_code_iso2",
    how="left"
)

print(f"Shape after join: {loans_enriched.shape}")
print(f"Unmatched rows: {loans_enriched['income_level'].isna().sum()}")

Shape after join: (11236, 35)
Unmatched rows: 662


What is the ratio of disbursed vs original principal — are loans being fully used?

In [5]:
total_principal = loans_enriched["original_principal_amount_usd"].sum()
total_disbursed = loans_enriched["disbursed_amount_usd"].sum()
total_undisbursed = loans_enriched["undisbursed_amount_usd"].sum()
total_cancelled = loans_enriched["cancelled_amount_usd"].sum()

print(f"Total Principal:   ${total_principal:,.0f}")
print(f"Total Disbursed:   ${total_disbursed:,.0f}  ({total_disbursed/total_principal*100:.1f}%)")
print(f"Total Undisbursed: ${total_undisbursed:,.0f}  ({total_undisbursed/total_principal*100:.1f}%)")
print(f"Total Cancelled:   ${total_cancelled:,.0f}  ({total_cancelled/total_principal*100:.1f}%)")

Total Principal:   $618,424,383,805
Total Disbursed:   $467,085,886,208  (75.5%)
Total Undisbursed: $106,801,114,203  (17.3%)
Total Cancelled:   $45,607,054,201  (7.4%)


In [7]:
disbursement = loans_enriched.groupby("income_level").agg(
    total_principal=("original_principal_amount_usd", "sum"),
    total_disbursed=("disbursed_amount_usd", "sum"),
    total_undisbursed=("undisbursed_amount_usd", "sum"),
    total_cancelled=("cancelled_amount_usd", "sum")
).reset_index()

disbursement["disbursed_pct"]   = (disbursement["total_disbursed"]   / disbursement["total_principal"] * 100).round(2)
disbursement["undisbursed_pct"] = (disbursement["total_undisbursed"] / disbursement["total_principal"] * 100).round(2)
disbursement["cancelled_pct"]   = (disbursement["total_cancelled"]   / disbursement["total_principal"] * 100).round(2)

print(disbursement[["income_level", "disbursed_pct", "undisbursed_pct", "cancelled_pct"]].to_string())

          income_level  disbursed_pct  undisbursed_pct  cancelled_pct
0          High income          71.67            23.22           6.41
1           Low income          72.53            22.32           5.25
2  Lower middle income          74.99            16.50           8.61
3       Not classified          77.94            15.80           5.92
4  Upper middle income          91.56             6.89           2.92


Which countries have the highest cancellation rates — money borrowed but never used?

In [24]:
disbursement = loans_enriched.groupby("country").agg(
    total_principal=("original_principal_amount_usd", "sum"),
    total_disbursed=("disbursed_amount_usd", "sum"),
    total_undisbursed=("undisbursed_amount_usd", "sum"),
    total_cancelled=("cancelled_amount_usd", "sum")
).reset_index()

disbursement["disbursed_pct"]   = (disbursement["total_disbursed"]   / disbursement["total_principal"] * 100).round(2)
disbursement["undisbursed_pct"] = (disbursement["total_undisbursed"] / disbursement["total_principal"] * 100).round(2)
disbursement["cancelled_pct"]   = (disbursement["total_cancelled"]   / disbursement["total_principal"] * 100).round(2)

top10 = (
    disbursement
    .sort_values(by="cancelled_pct", ascending=False)
    .head(10)
)
print(top10[["country", "disbursed_pct", "undisbursed_pct", "cancelled_pct"]].to_string(index=False))

    country  disbursed_pct  undisbursed_pct  cancelled_pct
Timor-Leste          34.48            13.25          51.57
    Myanmar          56.70             2.67          40.80
   Zimbabwe          72.43             0.00          27.65
       Iraq          73.89             0.00          26.75
  Caribbean          68.37            11.78          23.47
Philippines          82.94             0.00          18.60
      Sudan          78.15             7.15          16.17
    Morocco          84.48             0.00          15.52
 Costa Rica          85.23             0.00          14.77
   Viet Nam          84.69             1.93          12.80


Which income groups carry the most debt relative to what they actually received?

In [33]:
indebt = loans_enriched.groupby("income_level").agg(
    total_indebt=("due_to_ida_usd", "sum"),
    total_disbursed=("disbursed_amount_usd", "sum"),
).reset_index()

indebt["indebt_pct"]   = (indebt["total_indebt"]   / indebt["total_disbursed"] * 100).round(2)
top10 = (
    indebt
    .sort_values(by="indebt_pct", ascending=False)
)
print(top10[["income_level", "total_disbursed", "total_indebt", "indebt_pct"]].to_string(index=False))


       income_level  total_disbursed  total_indebt  indebt_pct
Lower middle income     2.859772e+11  1.650222e+11       57.70
     Not classified     2.996013e+10  1.433996e+10       47.86
Upper middle income     3.420836e+10  1.324139e+10       38.71
         Low income     9.269343e+10  3.119012e+10       33.65
        High income     6.780938e+08  1.984138e+08       29.26


What does the repayment landscape look like — how many loans are fully repaid vs still active vs cancelled?

In [37]:
status_map = {
    "Fully Repaid":          "Repaid",
    "Repaying":              "Active",
    "Disbursing":            "Active",
    "Fully Disbursed":       "Active",
    "Disbursing&Repaying":   "Active",
    "Effective":             "Active",
    "Approved":              "Active",
    "Signed":                "Active",
    "Fully Cancelled":       "Cancelled",
    "Terminated":            "Terminated"
}

loans_enriched["loan_bucket"] = loans_enriched["credit_status"].map(status_map)

bucket = loans_enriched.groupby("loan_bucket").agg(
    loan_count=("credit_number", "count"),
    total_principal=("original_principal_amount_usd", "sum")
).reset_index()

bucket["count_pct"] = (bucket["loan_count"] / bucket["loan_count"].sum() * 100).round(2)
bucket["value_pct"] = (bucket["total_principal"] / bucket["total_principal"].sum() * 100).round(2)

bucket["total_principal"] = bucket["total_principal"].apply(lambda x: f"${x:,.0f}")

print(bucket[["loan_bucket", "loan_count", "count_pct", "total_principal", "value_pct"]].to_string(index=False))

loan_bucket  loan_count  count_pct  total_principal  value_pct
     Active        8463      75.32 $548,567,992,382      88.70
  Cancelled         134       1.19   $5,598,684,001       0.91
     Repaid        2596      23.10  $63,530,407,423      10.27
 Terminated          43       0.38     $727,300,000       0.12


>The World Bank currently has $548B in active loans outstanding , 88% of everything it has ever committed is still in motion. Only 10% by value >has been fully repaid.

Are soft loan countries (IDA/XDR) borrowing more than they can handle?

We have two signals for this:

>Low disbursement rate :they can't even absorb the money they borrowed

>High debt burden : what they still owe is large relative to what they received

In [43]:
ida_countries = loans_enriched[loans_enriched["lending_type"] == "IDA"]

drowning_in_debt = ida_countries.groupby("country").agg(
    total_owed=("due_to_ida_usd", "sum"),
    total_disbursed=("disbursed_amount_usd", "sum"),
    total_principal=("original_principal_amount_usd", "sum"),
    total_cancelled=("cancelled_amount_usd", "sum")
).reset_index()

drowning_in_debt["debt_pct"] = (drowning_in_debt["total_owed"] / drowning_in_debt["total_disbursed"] * 100).round(2)
drowning_in_debt["cancelled_pct"] = (drowning_in_debt["total_cancelled"] / drowning_in_debt["total_principal"] * 100).round(2)

top10 = drowning_in_debt.sort_values("debt_pct", ascending=False).head(10)
print(top10[["country", "total_disbursed", "total_owed", "debt_pct", "cancelled_pct"]].to_string(index=False))

   country  total_disbursed   total_owed  debt_pct  cancelled_pct
   Eritrea     5.100836e+08 4.237235e+08     83.07          11.33
  Cambodia     2.418007e+09 1.907722e+09     78.90           2.73
    Kosovo     5.287890e+08 4.156607e+08     78.61           5.20
  Tanzania     1.973032e+10 1.407467e+10     71.34           3.21
Bangladesh     3.195738e+10 2.238269e+10     70.04          12.42
    Bhutan     6.297506e+08 4.295017e+08     68.20           1.10
   Senegal     7.828429e+09 5.175596e+09     66.11           4.22
     Nepal     7.695517e+09 5.026976e+09     65.32          12.45
   Lesotho     9.479199e+08 5.792158e+08     61.10           6.25
     Benin     4.653395e+09 2.810947e+09     60.41           3.07


Yes, IDA countries are borrowing more than many can handle. The soft loan terms make borrowing attractive, but the sheer volume of debt accumulating means repayment will consume significant portions of these countries' future budgets for decades.

In [48]:
morocco = loans_enriched[loans_enriched["country"] == "Morocco"]

total_principal = morocco["original_principal_amount_usd"].sum()
total_disbursed = morocco["disbursed_amount_usd"].sum()
total_cancelled = morocco["cancelled_amount_usd"].sum()
total_owed      = morocco["due_to_ida_usd"].sum()
loan_count      = morocco["credit_number"].count()

print(f"Income level:  {morocco['income_level'].iloc[0]}")
print(f"Lending type:  {morocco['lending_type'].iloc[0]}")
print(f"Loan count:    {loan_count}")
print(f"Total principal: ${total_principal:,.0f}")
print(f"Total disbursed: ${total_disbursed:,.0f}  ({total_disbursed/total_principal*100:.2f}%)")
print(f"Total cancelled: ${total_cancelled:,.0f}  ({total_cancelled/total_principal*100:.2f}%)")
print(f"Debt burden:     ${total_owed:,.0f}  ({total_owed/total_disbursed*100:.2f}%)")

Income level:  Lower middle income
Lending type:  IBRD
Loan count:    5
Total principal: $53,458,951
Total disbursed: $45,161,065  (84.48%)
Total cancelled: $8,297,887  (15.52%)
Debt burden:     $0  (0.00%)


In [47]:
print(morocco[["credit_number", "credit_status", "original_principal_amount_usd", 
               "disbursed_amount_usd", "repaid_to_ida_usd"]].to_string())

    credit_number credit_status  original_principal_amount_usd  disbursed_amount_usd  repaid_to_ida_usd
78       IDA00790  Fully Repaid                    12752937.42           11911688.58        11911688.58
171      IDA01670  Fully Repaid                     8206013.77            8206013.77         8206013.77
279      IDA02660      Repaying                     8500000.00            8500000.00         8500000.00
354      IDA03380      Repaying                    10000000.00           10000000.00        10000000.00
584      IDA05550  Fully Repaid                    14000000.00            6543362.22         6543362.22


### Layer 2 : The Loan Itself: Findings Summary

#### Core Question
> Are loans actually being used well — and are the poorest countries
> ending up with more debt than benefit?


##### Q1 — Disbursement vs Principal: Are Loans Being Fully Used?

**Global picture:**
| Bucket | Amount | Percentage |
|---|---|---|
| Total Principal Promised | 618 billion USD | 100% |
| Actually Disbursed | 467 billion USD | 75.5% |
| Still Pending | 106 billion USD | 17.3% |
| Cancelled | 45 billion USD | 7.4% |

**By income group:**
| Income Level | Disbursed % | Undisbursed % | Cancelled % |
|---|---|---|---|
| Upper middle income | 91.56% | 6.89% | 2.92% |
| Not classified | 77.94% | 15.80% | 5.92% |
| Lower middle income | 74.99% | 16.50% | 8.61% |
| Low income | 72.53% | 22.32% | 5.25% |
| High income | 71.67% | 23.22% | 6.41% |

**Key insight:**
Upper middle income countries are the most efficient borrowers —
they absorb 91% of what they are promised and rarely cancel.
Low income countries struggle to absorb even 73% of their loans,
leaving 22% pending and unused.

The paradox: the countries that need the money most are the least
able to absorb it efficiently — because they lack the institutional
capacity, project management infrastructure, and stable governance
to execute large projects.


##### Q2 — Which Countries Have the Highest Cancellation Rates?

**Top 5 by cancellation rate:**
| Country | Disbursed % | Cancelled % | Reason |
|---|---|---|---|
| Timor-Leste | 34.48% | 51.57% | Extreme institutional weakness — one of world's newest nations |
| Myanmar | 56.70% | 40.80% | Military coup 2021 — World Bank suspended all operations |
| Zimbabwe | 72.43% | 27.65% | Economic collapse, hyperinflation, governance failure |
| Iraq | 73.89% | 26.75% | Post-war instability and corruption |
| Caribbean | 68.37% | 23.47% | Small island states — limited implementation capacity |

**Two distinct cancellation patterns:**
1. Political and conflict collapse — Myanmar, Iraq, Zimbabwe, Sudan
   The World Bank cancelled not because projects failed but because
   the country itself destabilized

2. Institutional weakness — Timor-Leste, Caribbean, Philippines
   The country simply could not execute — too young, too small,
   or too under-resourced

**Key insight:**
Cancellation is not a financial loss for either side — the country
does not receive the money and does not owe it back. But the real
cost is the development that never happened. Countries spent years
negotiating loans, signed agreements, set up project teams — and
ended up with nothing but lost time and the same unresolved problems.

##### Q3 — Which Income Groups Carry the Most Debt?

**Debt burden as percentage of what was received:**
| Income Level | Still Owed % |
|---|---|
| Lower middle income | 57.70% |
| Not classified | 47.86% |
| Upper middle income | 38.71% |
| Low income | 33.65% |
| High income | 29.26% |

**Key insight — the surprise:**
Low income countries are NOT the most indebted group.
Lower middle income countries carry the heaviest burden.

Why? Low income countries receive IDA grants and zero interest
loans — a portion is never expected to be fully repaid, keeping
their ratio lower. Lower middle income countries get harder mixed
terms with higher rates and longer repayment periods.

The debt trap is hitting the middle tier — countries like Nigeria,
Pakistan, and Bangladesh. Just developed enough to get harder loan
terms, but not developed enough to repay quickly.


##### Q4 — The Repayment Landscape

**Loan status breakdown:**
| Status | Loan Count | Count % | Total Principal | Value % |
|---|---|---|---|---|
| Active | 8,463 | 75.32% | 548 billion USD | 88.70% |
| Repaid | 2,596 | 23.10% | 63 billion USD | 10.27% |
| Cancelled | 134 | 1.19% | 5.6 billion USD | 0.91% |
| Terminated | 43 | 0.38% | 727 million USD | 0.12% |

**Key insight:**
88% of everything the World Bank has ever committed — 548 billion USD —
is still active and outstanding. Only 10% by value has been fully repaid.

This means the full debt impact on developing countries is yet to be
felt. Most of the repayment burden is still ahead of them. The real
test of whether these loans were worth it will play out over the
next 20 to 40 years.


##### Q5 — Are Soft Loan Countries Borrowing More Than They Can Handle?

**Top IDA countries by debt burden:**
| Country | Received | Still Owes | Debt % |
|---|---|---|---|
| Eritrea | 510 million USD | 424 million USD | 83.07% |
| Cambodia | 2.4 billion USD | 1.9 billion USD | 78.90% |
| Kosovo | 529 million USD | 416 million USD | 78.61% |
| Tanzania | 19.7 billion USD | 14.1 billion USD | 71.34% |
| Bangladesh | 31.9 billion USD | 22.4 billion USD | 70.04% |

**Two types of struggling IDA countries:**

Small fragile states — Eritrea, Kosovo, Lesotho, Bhutan
Tiny economies carrying debt loads massive relative to their size.
Eritrea still owes 83 cents of every dollar it ever received.

Large high-borrowing nations — Bangladesh, Tanzania, Nepal
These countries borrowed heavily over decades and have accumulated
debt that will consume significant portions of their future budgets.

**The Morocco exception:**
Morocco is a rare success story — borrowed from IDA when poor,
repaid everything, and graduated to IBRD as its economy developed.
This proves loans can work when a country has decent institutions
and governance. It is the counterpoint to the broader narrative.

**Answer to Q5:**
Yes — many IDA countries are borrowing more than they can handle.
The soft loan terms make borrowing attractive and accessible, but
the sheer volume of accumulated debt means repayment will consume
future budgets for decades. The softest loans go to the countries
least equipped to use them efficiently and most vulnerable to the
long term repayment burden.

#### Layer 2 — Overall Verdict

1. 75 cents of every dollar promised actually reaches its destination —
   but upper middle income countries absorb loans far more efficiently
   than low income ones

2. Cancellations hit the most fragile states hardest — conflict,
   instability, and weak institutions turn borrowed opportunities
   into wasted years

3. The debt trap is not hitting the very poorest — it is squeezing
   the middle tier of developing countries hardest

4. 88% of all World Bank loan value is still active — the real
   repayment burden has not yet arrived for most borrowers

5. IDA soft loans are necessary but not sufficient — giving poor
   countries access to cheap money does not solve the deeper problem
   of institutional capacity to use it well

> The loan is not the problem. The system around the loan —
> who gets the contracts, whether projects get executed,
> whether institutions are strong enough to absorb the money —
> that is where developing countries win or lose.